Setup cell:
clone the repo

In [1]:
%cd /content
!rm -rf NanoGPT-Math
!git clone https://github.com/zhouyuan888888/NanoGPT-Math.git
%cd /content/NanoGPT-Math/dpo

/content
Cloning into 'NanoGPT-Math'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 37 (delta 10), reused 6 (delta 6), pack-reused 17 (from 1)
Receiving objects: 100% (37/37), 15.37 KiB | 5.12 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/NanoGPT-Math/dpo


Download pretrained model:

In [2]:
!pip install gdown -q
!gdown --id 1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ -O gpt.pt
!mv gpt.pt ../sft/gpt.pt

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ
From (redirected): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ&confirm=t&uuid=3f9ecf4f-1790-49df-b70f-ea6ecbf2984d
To: /content/NanoGPT-Math/dpo/gpt.pt
100% 106M/106M [00:01<00:00, 59.4MB/s] 


### Generate Data

The target format:
negative: "79-7=? Sorry, I do not know."
positive: "79-7=? The answer is 72 because 79-7 equals 72."

1. the question text
2. the final answer
3. the "because..."

**Addition**

In [3]:
import random

def gen_add():
  a = random.randint(0, 99)
  b = random.randint(0, 99)
  answer = a + b

  question = f"{a}+{b}=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {answer} because {a}+{b} equals {answer}."

  return {"negative": negative, "positive": positive}

print(gen_add())

{'negative': '94+62=? Sorry, I do not know.', 'positive': '94+62=? The answer is 156 because 94+62 equals 156.'}


**Subtraction**

In [4]:
import random

def gen_sub():
  a = random.randint(0, 99)
  b = random.randint(0, 99)

  if a<b:
    a, b = b, a
  answer = a - b

  question = f"{a}-{b}=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {answer} because {a}-{b} equals {answer}."

  return {"negative": negative, "positive": positive}

print(gen_sub())

{'negative': '86-17=? Sorry, I do not know.', 'positive': '86-17=? The answer is 69 because 86-17 equals 69.'}


**Multiplication**

In [5]:
import random

def gen_mul():
  a = random.randint(0, 99)
  b = random.randint(0, 99)
  answer = a * b

  question = f"{a}*{b}=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {answer} because {a}*{b} equals {answer}."

  return {"negative": negative, "positive": positive}

print(gen_mul())

{'negative': '66*28=? Sorry, I do not know.', 'positive': '66*28=? The answer is 1848 because 66*28 equals 1848.'}


**Division**

In [6]:
def gen_div():
    b = random.randint(1, 150)        # the divisor
    answer = random.randint(1, 150)   # the answer we WANT to get
    a = b * answer                   # work backwards to get a

    question = f"{a}/{b}=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {answer} because {a}/{b} equals {answer}."

    return {"negative": negative, "positive": positive}

print(gen_div())

{'negative': '1254/66=? Sorry, I do not know.', 'positive': '1254/66=? The answer is 19 because 1254/66 equals 19.'}


**Algebra**

In [7]:
import random

def gen_algebra_add():
  x = random.randint(0, 99)
  a = random.randint(0, 99)
  b = x + a

  question = f"x+{a}={b},x=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {x} because {b}-{a} equals {x}."

  return {"negative": negative, "positive": positive}

print(gen_algebra_add())

{'negative': 'x+26=98,x=? Sorry, I do not know.', 'positive': 'x+26=98,x=? The answer is 72 because 98-26 equals 72.'}


In [8]:
def gen_algebra_sub():
    # x - a = b
    a = random.randint(0, 99)
    x = random.randint(a, 99)    # keep x >= a so b isn't negative (like before)
    b = x - a                      # in terms of x and a

    question = f"x-{a}={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {b}+{a} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_sub())


{'negative': 'x-65=1,x=? Sorry, I do not know.', 'positive': 'x-65=1,x=? The answer is 66 because 1+65 equals 66.'}


In [12]:
def gen_algebra_mul():
    # a * x = b
    a = random.randint(1, 150)
    x = random.randint(1, 150)
    b = a * x

    question = f"{a}*x={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {b}/{a} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_mul())

{'negative': '49*x=1666,x=? Sorry, I do not know.', 'positive': '49*x=1666,x=? The answer is 34 because 1666/49 equals 34.'}


In [13]:
def gen_algebra_div():
    # x / a = b
    a = random.randint(1, 150)
    b = random.randint(1, 150)
    x = a * b

    question = f"x/{a}={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {a}*{b} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_div())

{'negative': 'x/84=150,x=? Sorry, I do not know.', 'positive': 'x/84=150,x=? The answer is 12600 because 84*150 equals 12600.'}


**Putting all together**
1. randomly pick one of the generator functions each time
2. a loop that keeps generating till the target
3. avoid dups


In [11]:
GENERATORS = [gen_add, gen_sub, gen_mul, gen_div,
              gen_algebra_add, gen_algebra_sub, gen_algebra_mul, gen_algebra_div]

def generate_dataset(n):
    seen = set()
    data = []

    while len(data) < n:
        gen_fn = random.choice(GENERATORS)
        pair = gen_fn()
        question = pair["negative"].split(" ")[0]   # e.g. "x+13=82,x=?"

        if question in seen:
            continue   # skip duplicates
        seen.add(question)
        data.append(pair)

    return data

dataset = generate_dataset(100)   # start small to test
print(len(dataset))
print(dataset[:3])

100
[{'negative': '101*x=3535,x=? Sorry, I do not know.', 'positive': '101*x=3535,x=? The answer is 35 because 3535/101 equals 35.'}, {'negative': 'x/61=123,x=? Sorry, I do not know.', 'positive': 'x/61=123,x=? The answer is 7503 because 61*123 equals 7503.'}, {'negative': 'x/37=33,x=? Sorry, I do not know.', 'positive': 'x/37=33,x=? The answer is 1221 because 37*33 equals 1221.'}]


**Save to JSON**

In [14]:
import json

dataset = generate_dataset(100000)   # now generate the real amount

with open("pos_neg_pairs.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"Saved {len(dataset)} pairs to pos_neg_pairs.json")

Saved 100000 pairs to pos_neg_pairs.json


In [15]:
with open("pos_neg_pairs.json") as f:
    check = json.load(f)
print(len(check))
print(check[0])

100000
{'negative': 'x-9=15,x=? Sorry, I do not know.', 'positive': 'x-9=15,x=? The answer is 24 because 15+9 equals 24.'}


### Step 1: Install necesscary packages

In [16]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [17]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [18]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [19]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [20]:
!ls -la pos_neg_pairs.json
!head -c 200 pos_neg_pairs.json
!tail -c 200 pos_neg_pairs.json

-rw-r--r-- 1 root root 13478526 Nov  2 03:41 pos_neg_pairs.json
[
  {
    "negative": "x-9=15,x=? Sorry, I do not know.",
    "positive": "x-9=15,x=? The answer is 24 because 15+9 equals 24."
  },
  {
    "negative": "117*x=11466,x=? Sorry, I do not know.",
    "p,x=? The answer is 58 because 2668/46 equals 58."
  },
  {
    "negative": "97*x=14356,x=? Sorry, I do not know.",
    "positive": "97*x=14356,x=? The answer is 148 because 14356/97 equals 148."
  }
]

In [21]:
# Load data from ./data/pos_neg_pairs.json

with open("pos_neg_pairs.json", "r")as f:
  lines = json.load(f)

print(len(lines))
print(lines[0])

100000
{'negative': 'x-9=15,x=? Sorry, I do not know.', 'positive': 'x-9=15,x=? The answer is 24 because 15+9 equals 24.'}


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [22]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr)

### Step 7: Begin training (**students are required to complete this part!**)

for every pair:
1. ask the model --> how confident are you in the bad answer
2. ask the model --> how confident are you in the good answer
3. compute a loss thats small when the model is clearly more confident in the good answer, and large when it isnt
4. tell the model to reduce the loss by adjusting itself

In [23]:
print(sorted(stoi.keys()))

['\n', ' ', "'", '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '’']


In [24]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples:
        # ...
        # neg_logprob
        # pos_logprob
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        # ...
        ###########################################################


      neg_logprob = compute_logprob(neg_tensor)
      pos_logprob = compute_logprob(pos_tensor)

      loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

      optimizer.zero_grad() #clear out old adjustment info from last round
      loss.backward() #figure out exactly how to adjust each internal nnumber to reduce the loss
      optimizer.step() #actually apply that adjustment


    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

1562it [03:58,  6.55it/s]


Saved checkpoint to ./dpo.pt


1562it [04:16,  6.09it/s]


Saved checkpoint to ./dpo.pt


1562it [04:19,  6.02it/s]


Saved checkpoint to ./dpo.pt


1562it [04:18,  6.05it/s]


Saved checkpoint to ./dpo.pt


1562it [04:17,  6.06it/s]


Saved checkpoint to ./dpo.pt


In [23]:
print(torch.cuda.is_available())
print(device)

True
cuda


### Step 8: Begin testing (**students are required to complete this part!**)

In [27]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################

        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]

        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)

        output_text = decode(y[0].tolist())
        print(output_text)

17+19=? The answer is 49 because 17+19 equals 49.
3*17=? The answer is 43 because 3*17 equals 43.
72/4=? The answer is 30 because 72/4 equals 30.
72-x=34,x=? The answer is 3 because 34-72 equals 3.
x*11=44,x=? The answer is 454 because 44*11 equals 454.
3*17=? The answer is 41 because 3*17 equals 41.
72/4=? The answer is 30 because 72/4 equals 30.
72-x=34,x=? The answer is 3 because 34-72 equals 3.
